# AutoML

In [15]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import dotenv
import os

dotenv.load_dotenv(override=True)

ml_client = ml_client = MLClient(
     credential         = DefaultAzureCredential()
    ,subscription_id    = os.environ["AZURE_SUBSCRIPTION_ID"]
    ,resource_group_name= os.environ["AZURE_RESOURCE_GROUP"]
    ,workspace_name     = os.environ["AZURE_ML"]
)

compute_cluster_name = "AutoML-cluster"
compute_cluster_size = "STANDARD_DS3_v2"

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


In [16]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

data    = "diabetes-data-ml-table"
version = "0"

training_data_input = Input(type=AssetTypes.MLTABLE, path=f"azureml:{data}:{version}")

One of the most important settings you must specify is the primary_metric. The primary metric is the target performance metric for which the optimal model will be determined. To retrieve the list of metrics available when you want to train a classification model, you can use the ```ClassificationPrimaryMetrics``` function as shown here:

In [17]:
from azure.ai.ml import automl

# configure the classification job
classification_job = automl.classification(
    compute                         = compute_cluster_name,
    experiment_name                 = "auto-ml-class-dev",
    training_data                   = training_data_input,
    target_column_name              = "Diabetic",
    primary_metric                  = "accuracy",
    n_cross_validations             = 5,
    enable_model_explainability     = True
)

To see which primary metrics are available:

In [18]:
from azure.ai.ml.automl import ClassificationPrimaryMetrics
 
list(ClassificationPrimaryMetrics)

[<ClassificationPrimaryMetrics.AUC_WEIGHTED: 'AUCWeighted'>,
 <ClassificationPrimaryMetrics.ACCURACY: 'Accuracy'>,
 <ClassificationPrimaryMetrics.NORM_MACRO_RECALL: 'NormMacroRecall'>,
 <ClassificationPrimaryMetrics.AVERAGE_PRECISION_SCORE_WEIGHTED: 'AveragePrecisionScoreWeighted'>,
 <ClassificationPrimaryMetrics.PRECISION_SCORE_WEIGHTED: 'PrecisionScoreWeighted'>]

Training machine learning models will cost compute. To minimize costs and time spent on training, you can set limits to an AutoML experiment or job by using ```set_limits()```:
- ```timeout_minutes```: Number of minutes after which the complete AutoML experiment is terminated.
- ```trial_timeout_minutes```: Maximum number of minutes one trial can take.
- ```max_trials```: Maximum number of trials, or models that will be trained.
- ```enable_early_termination```: Whether to end the experiment if the score isn't improving in the short term.

You can use ```max_concurrent_trials``` to set the maximum number of parallel trials to be less than the maximum number of nodes.

In [19]:
classification_job.set_limits(
    timeout_minutes=30, 
    trial_timeout_minutes=10, 
    max_trials=5,
    enable_early_termination=True,
)

In [23]:
from azure.ai.ml.entities import AmlCompute

if compute_cluster_name not in [c.name for c in ml_client.compute.list()]:

    cc = AmlCompute(
        name = compute_cluster_name
        ,size = compute_cluster_size
        ,idle_time_before_scale_down=15
        ,location="westeurope"
        ,min_instances=0
        ,max_instances=2
        ,tier="low_priority"
    )

    poller = ml_client.compute.begin_create_or_update(cc)

else:
    cc = ml_client.compute.get(compute_cluster_name)

Run autoML classification job

In [22]:
returned_job = ml_client.jobs.create_or_update(
    classification_job
) 
print(returned_job.studio_url)

https://ml.azure.com/runs/goofy_root_40f9967g77?wsid=/subscriptions/e46b15c2-60e4-419f-a352-ee019a3b0daf/resourcegroups/dg-associate-data-science/workspaces/aml-dg-associate-data-science&tid=d018aec4-2b2b-4c66-9939-2c96877e6bf1


In [24]:
ml_client.compute.begin_delete(cc)

HttpResponseError: Operation returned an invalid status 'Bad Request'
Content: <!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN""http://www.w3.org/TR/html4/strict.dtd">
<HTML><HEAD><TITLE>Bad Request</TITLE>
<META HTTP-EQUIV="Content-Type" Content="text/html; charset=us-ascii"></HEAD>
<BODY><h2>Bad Request - Invalid URL</h2>
<hr><p>HTTP Error 400. The request URL is invalid.</p>
</BODY></HTML>
